In [1]:
import subprocess, torch, json, platform
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,power.limit","--format=csv"],capture_output=True,text=True).stdout)
p = torch.cuda.get_device_properties(0)
print(json.dumps({"gpu": p.name, "vram_gb": round(p.total_memory/1e9,1), "capability": f"{p.major}.{p.minor}",
  "bf16": torch.cuda.is_bf16_supported(), "torch": torch.__version__, "cuda": torch.version.cuda,
  "python": platform.python_version()}, indent=2))


name, memory.total [MiB], driver_version, power.limit [W]
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07, 400.00 W

{
  "gpu": "NVIDIA A100-SXM4-40GB",
  "vram_gb": 42.4,
  "capability": "8.0",
  "bf16": true,
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "python": "3.13.15"
}


In [2]:
# Block 1 — environment, results folder, env.json
!pip install -q transformers accelerate nvidia-ml-py

import os, json, platform, subprocess, datetime
import torch, transformers, accelerate

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/gpu_bench/a100_40gb'
except Exception as e:
    print("Drive not mounted -> saving locally only:", e)
    RESULTS_DIR = '/content/gpu_bench/a100_40gb'
os.makedirs(RESULTS_DIR, exist_ok=True)

p = torch.cuda.get_device_properties(0)
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,power.limit,clocks.max.sm",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
env = {
    "gpu": p.name, "vram_gb": round(p.total_memory / 2**30, 1), "capability": f"{p.major}.{p.minor}",
    "sm_count": p.multi_processor_count, "nvidia_smi": smi,
    "torch": torch.__version__, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "python": platform.python_version(), "cpu_count": os.cpu_count(),
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
json.dump(env, open(f"{RESULTS_DIR}/env.json", "w"), indent=2)
print(json.dumps(env, indent=2))
print("Results ->", RESULTS_DIR)


Drive not mounted -> saving locally only: [dfs_ephemeral] Credentials propagation unsuccessful: the server still reports the credentials as unauthorized
{
  "gpu": "NVIDIA A100-SXM4-40GB",
  "vram_gb": 39.5,
  "capability": "8.0",
  "sm_count": 108,
  "nvidia_smi": "NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07, 400.00 W, 1410 MHz",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "cudnn": 92700,
  "transformers": "5.18.0",
  "accelerate": "1.15.0",
  "python": "3.13.15",
  "cpu_count": 12,
  "timestamp": "2026-10-14T00:00:57"
}
Results -> /content/gpu_bench/a100_40gb


In [3]:
# Block 2 — shared helpers: GPU sampler, timing, resumable result saving
import time, threading, csv, gc, contextlib
import pynvml

pynvml.nvmlInit()
_H = pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME = torch.cuda.get_device_name(0)

# A100 dense peak TFLOPS, used for an approximate MFU column
PEAK_TFLOPS = {"fp32": 19.5, "tf32": 156.0, "fp16": 312.0, "bf16": 312.0}

class GPUSampler:
    """Samples util / power / memory every `interval` s in a background thread."""
    def __init__(self, interval=0.1):
        self.interval, self.util, self.power, self.mem = interval, [], [], []
        self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.util.append(pynvml.nvmlDeviceGetUtilizationRates(_H).gpu)
            self.power.append(pynvml.nvmlDeviceGetPowerUsage(_H) / 1000)
            self.mem.append(pynvml.nvmlDeviceGetMemoryInfo(_H).used / 2**30)
            time.sleep(self.interval)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join()
    def stats(self):
        avg = lambda x: round(sum(x) / len(x), 1) if x else None
        return {"gpu_util_avg": avg(self.util), "gpu_util_max": max(self.util, default=None),
                "power_avg_w": avg(self.power), "power_max_w": round(max(self.power, default=0), 1),
                "nvml_mem_max_gb": round(max(self.mem, default=0), 2)}

def cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def set_tf32(on):
    torch.set_float32_matmul_precision("high" if on else "highest")
    torch.backends.cudnn.allow_tf32 = on

def amp_ctx(prec):
    if prec == "fp16": return torch.autocast("cuda", dtype=torch.float16)
    if prec == "bf16": return torch.autocast("cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()          # fp32 / tf32

def run_timed(step_fn, warmup, iters):
    """Warm up, then time `iters` steps. Returns seconds, GPU stats, peak alloc/reserved GB."""
    for _ in range(warmup): step_fn()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    with GPUSampler() as s:
        t0 = time.perf_counter()
        for _ in range(iters): step_fn()
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0
    return (dt, s.stats(),
            round(torch.cuda.max_memory_allocated() / 2**30, 2),
            round(torch.cuda.max_memory_reserved() / 2**30, 2))

def _path(name): return f"{RESULTS_DIR}/{name}.jsonl"

def done_keys(name):
    if not os.path.exists(_path(name)): return set()
    return {json.loads(l)["key"] for l in open(_path(name)) if l.strip()}

def save_result(name, row):
    """Append to JSONL (fsync'd), then rewrite the matching CSV."""
    with open(_path(name), "a") as f:
        f.write(json.dumps(row) + "\n"); f.flush(); os.fsync(f.fileno())
    rows = [json.loads(l) for l in open(_path(name)) if l.strip()]
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with open(f"{RESULTS_DIR}/{name}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)

def base_row(key, **kw):
    return {"key": key, "gpu": GPU_NAME, "torch": torch.__version__,
            "ts": datetime.datetime.now().isoformat(timespec="seconds"), **kw}

with GPUSampler() as s: time.sleep(0.5)
print("Sampler OK:", s.stats())


Sampler OK: {'gpu_util_avg': 0.0, 'gpu_util_max': 0, 'power_avg_w': 44.0, 'power_max_w': 44.0, 'nvml_mem_max_gb': 0.51}


In [4]:
# Block 3 — GPT-2 124M training (synthetic tokens, seq 1024, AdamW)
from transformers import GPT2Config, GPT2LMHeadModel
import pandas as pd

WARMUP, ITERS, SEQ = 10, 40, 1024
PRECISIONS = ["fp32", "tf32", "fp16", "bf16"]   # fp32 = strict IEEE (TF32 off)
BATCHES = [8, 16, 32]

def gpt2_train(prec, bs):
    set_tf32(prec != "fp32")
    torch.manual_seed(0)
    model = GPT2LMHeadModel(GPT2Config(n_positions=SEQ)).cuda().train()
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=(prec == "fp16"))
    x = torch.randint(0, model.config.vocab_size, (bs, SEQ), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, WARMUP, ITERS)
    tok_s = bs * SEQ * ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12          # approx, excludes attention FLOPs
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

NAME = "train_gpt2"
done = done_keys(NAME)
for prec in PRECISIONS:
    for bs in BATCHES:
        key = f"gpt2-124m|{prec}|bs{bs}|seq{SEQ}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="gpt2-124m", task="train", precision=prec, batch=bs, seq_len=SEQ,
                       warmup=WARMUP, iters=ITERS, optimizer="AdamW(fused)", data="synthetic")
        try:
            row.update(gpt2_train(prec, bs))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME, row)
        print(f"{key:32s} {row['status']:4s} "
              f"{row.get('tokens_per_s', '-'):>8} tok/s  peak {row.get('peak_alloc_gb', '-')} GB  "
              f"util {row.get('gpu_util_avg', '-')}%  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME}.csv")
print(df[["precision", "batch", "status", "tokens_per_s", "step_ms", "peak_alloc_gb",
          "gpu_util_avg", "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


gpt2-124m|fp32|bs8|seq1024       ok      18479 tok/s  peak 14.65 GB  util 100.0%  346.6 W
gpt2-124m|fp32|bs16|seq1024      ok      18984 tok/s  peak 27.9 GB  util 100.0%  360.9 W
gpt2-124m|fp32|bs32|seq1024      OOM         - tok/s  peak - GB  util -%  - W
gpt2-124m|tf32|bs8|seq1024       ok      48612 tok/s  peak 14.65 GB  util 100.0%  356.5 W
gpt2-124m|tf32|bs16|seq1024      ok      50451 tok/s  peak 27.9 GB  util 100.0%  369.4 W
gpt2-124m|tf32|bs32|seq1024      OOM         - tok/s  peak - GB  util -%  - W
gpt2-124m|fp16|bs8|seq1024       ok      62553 tok/s  peak 12.63 GB  util 100.0%  306.1 W
gpt2-124m|fp16|bs16|seq1024      ok      65505 tok/s  peak 23.6 GB  util 100.0%  313.9 W
gpt2-124m|fp16|bs32|seq1024      OOM         - tok/s  peak - GB  util -%  - W
gpt2-124m|bf16|bs8|seq1024       ok      63687 tok/s  peak 12.63 GB  util 100.0%  294.6 W
gpt2-124m|bf16|bs16|seq1024      ok      66284 tok/s  peak 23.6 GB  util 100.0%  301.8 W
gpt2-124m|bf16|bs32|seq1024      OOM         - tok

In [5]:
# Block 4 — ResNet-50 training (synthetic 224x224, channels_last, SGD momentum)
import torchvision

print(pd.read_csv(f"{RESULTS_DIR}/train_gpt2.csv").query("status == 'ok'")[["precision", "attn_impl"]]
      .drop_duplicates().to_string(index=False), "\n")

WARMUP, ITERS, IMG = 10, 40, 224
RN_BATCHES = [64, 128, 256]
RN_TRAIN_FLOPS_PER_IMG = 3 * 2 * 4.1e9      # ~4.1 GMACs fwd; train ≈ 3x fwd (approx)
torch.backends.cudnn.benchmark = True

def resnet_train(prec, bs):
    set_tf32(prec != "fp32")
    torch.manual_seed(0)
    model = torchvision.models.resnet50(weights=None).cuda().to(memory_format=torch.channels_last).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(prec == "fp16"))
    loss_fn = torch.nn.CrossEntropyLoss()
    x = torch.randn(bs, 3, IMG, IMG, device="cuda").to(memory_format=torch.channels_last)
    y = torch.randint(0, 1000, (bs,), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, WARMUP, ITERS)
    img_s = bs * ITERS / dt
    tflops = RN_TRAIN_FLOPS_PER_IMG * img_s / 1e12
    return {"status": "ok", "images_per_s": round(img_s, 1), "step_ms": round(1000 * dt / ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "images_per_joule": round(img_s / gpu["power_avg_w"], 2) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1)}

NAME_RN = "train_resnet50"
done = done_keys(NAME_RN)
for prec in PRECISIONS:
    for bs in RN_BATCHES:
        key = f"resnet50|{prec}|bs{bs}|img{IMG}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="resnet50", task="train", precision=prec, batch=bs, img_size=IMG,
                       warmup=WARMUP, iters=ITERS, optimizer="SGD(momentum)", data="synthetic",
                       memory_format="channels_last")
        try:
            row.update(resnet_train(prec, bs))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME_RN, row)
        print(f"{key:28s} {row['status']:4s} {row.get('images_per_s', '-'):>8} img/s  "
              f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_RN}.csv")
print(df[["precision", "batch", "status", "images_per_s", "step_ms", "peak_alloc_gb",
          "power_avg_w", "images_per_joule", "approx_mfu_pct"]].to_string(index=False))


precision attn_impl
     fp32      sdpa
     tf32      sdpa
     fp16      sdpa
     bf16      sdpa 

resnet50|fp32|bs64|img224    ok      444.2 img/s  peak 5.42 GB  310.4 W
resnet50|fp32|bs128|img224   ok      464.6 img/s  peak 10.58 GB  324.0 W
resnet50|fp32|bs256|img224   ok      479.5 img/s  peak 21.52 GB  331.5 W
resnet50|tf32|bs64|img224    ok     1040.4 img/s  peak 5.41 GB  309.1 W
resnet50|tf32|bs128|img224   ok     1075.8 img/s  peak 10.57 GB  323.4 W
resnet50|tf32|bs256|img224   ok     1103.4 img/s  peak 20.91 GB  327.6 W
resnet50|fp16|bs64|img224    ok     1870.3 img/s  peak 2.95 GB  272.3 W
resnet50|fp16|bs128|img224   ok     2188.6 img/s  peak 5.63 GB  316.0 W
resnet50|fp16|bs256|img224   ok     2277.9 img/s  peak 10.96 GB  330.7 W
resnet50|bf16|bs64|img224    ok     1747.2 img/s  peak 2.95 GB  282.4 W
resnet50|bf16|bs128|img224   ok     1868.7 img/s  peak 5.63 GB  299.3 W
resnet50|bf16|bs256|img224   ok     1943.7 img/s  peak 10.96 GB  311.9 W
precision  batch status  ima

In [6]:
# Block 5 — Qwen2.5-1.5B training (random init from config, bf16 AMP, fp32 master weights, fused AdamW)
from transformers import AutoConfig, AutoModelForCausalLM

QWEN = "Qwen/Qwen2.5-1.5B"
qcfg = AutoConfig.from_pretrained(QWEN)      # config.json only, no weights
qcfg.use_cache = False
Q_WARMUP, Q_ITERS = 5, 20
Q_CONFIGS = [            # (batch, seq_len, grad_checkpointing)
    (4, 512, True),
    (8, 512, True),
    (4, 1024, True),
    (2, 512, False),     # no checkpointing -> shows the recompute cost
]

def qwen_train(bs, seq, ckpt, prec="bf16"):
    set_tf32(True)
    torch.manual_seed(0)
    with torch.device("cuda"):
        model = AutoModelForCausalLM.from_config(qcfg, attn_implementation="sdpa")
    model.train()
    if ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, fused=True)
    x = torch.randint(0, qcfg.vocab_size, (bs, seq), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, Q_WARMUP, Q_ITERS)
    tok_s = bs * seq * Q_ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12          # model FLOPs only (recompute not counted)
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / Q_ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

NAME_Q = "train_qwen2.5-1.5b"
done = done_keys(NAME_Q)
for bs, seq, ckpt in Q_CONFIGS:
    key = f"qwen2.5-1.5b|bf16|bs{bs}|seq{seq}|ckpt{int(ckpt)}"
    if key in done:
        print("skip (done):", key); continue
    row = base_row(key, model="qwen2.5-1.5b", task="train", precision="bf16", batch=bs, seq_len=seq,
                   grad_ckpt=ckpt, warmup=Q_WARMUP, iters=Q_ITERS, optimizer="AdamW(fused)",
                   master_weights="fp32", data="synthetic", init="random(from config)")
    try:
        row.update(qwen_train(bs, seq, ckpt))
    except torch.cuda.OutOfMemoryError:
        row.update(status="OOM")
    cleanup()
    save_result(NAME_Q, row)
    print(f"{key:38s} {row['status']:4s} {row.get('tokens_per_s', '-'):>7} tok/s  "
          f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_Q}.csv")
print(df[["batch", "seq_len", "grad_ckpt", "status", "tokens_per_s", "step_ms", "peak_alloc_gb",
          "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

qwen2.5-1.5b|bf16|bs4|seq512|ckpt1     ok      9839 tok/s  peak 12.39 GB  321.8 W
qwen2.5-1.5b|bf16|bs8|seq512|ckpt1     ok     12008 tok/s  peak 15.98 GB  373.9 W
qwen2.5-1.5b|bf16|bs4|seq1024|ckpt1    ok     11793 tok/s  peak 15.98 GB  381.6 W
qwen2.5-1.5b|bf16|bs2|seq512|ckpt0     ok      8739 tok/s  peak 13.19 GB  280.7 W
 batch  seq_len  grad_ckpt status  tokens_per_s  step_ms  peak_alloc_gb  power_avg_w  tokens_per_joule  approx_mfu_pct
     4      512       True     ok          9839    208.2          12.39        321.8              30.6            29.2
     8      512       True     ok         12008    341.1          15.98        373.9              32.1            35.6
     4     1024       True     ok         11793    347.3          15.98        381.6              30.9            35.0
     2      512      False     ok          8739    117.2          13.19        280.7              31.1            25.9


In [7]:
# Block 5b — verify Qwen param dtype, relabel Block 5 rows, re-run with true fp32 master weights
with torch.device("meta"):
    _m = AutoModelForCausalLM.from_config(qcfg)
default_dtype = next(_m.parameters()).dtype; del _m
print("from_config default param dtype:", default_dtype)

def rewrite_results(name, rows):
    tmp = _path(name) + ".tmp"
    with open(tmp, "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, _path(name))
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with open(f"{RESULTS_DIR}/{name}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)

rows = [json.loads(l) for l in open(_path(NAME_Q)) if l.strip()]
stale = [r for r in rows if "param_dtype" not in r]          # rows written by Block 5
if stale:
    dt_name = str(default_dtype).replace("torch.", "")
    for r in stale:
        r["param_dtype"] = dt_name
        if default_dtype == torch.bfloat16:
            r.update(key=r["key"].replace("|bf16|", "|bf16-pure|"), precision="bf16-pure",
                     master_weights="none (bf16 weights + bf16 AdamW states)")
    rewrite_results(NAME_Q, rows)
    print(f"tagged {len(stale)} Block 5 rows with param_dtype={dt_name}")

def qwen_train_fp32master(bs, seq, ckpt, prec="bf16"):
    set_tf32(True)
    torch.manual_seed(0)
    with torch.device("cuda"):
        model = AutoModelForCausalLM.from_config(qcfg, attn_implementation="sdpa")
    model = model.float().train()                              # force fp32 master weights
    assert next(model.parameters()).dtype == torch.float32
    if ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5, fused=True)
    x = torch.randint(0, qcfg.vocab_size, (bs, seq), device="cuda")

    def step():
        with amp_ctx(prec):
            loss = model(input_ids=x, labels=x).loss
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)

    dt, gpu, peak_alloc, peak_res = run_timed(step, Q_WARMUP, Q_ITERS)
    tok_s = bs * seq * Q_ITERS / dt
    tflops = 6 * n_params * tok_s / 1e12
    return {"status": "ok", "tokens_per_s": round(tok_s), "step_ms": round(1000 * dt / Q_ITERS, 1),
            "peak_alloc_gb": peak_alloc, "peak_reserved_gb": peak_res, **gpu,
            "tokens_per_joule": round(tok_s / gpu["power_avg_w"], 1) if gpu["power_avg_w"] else None,
            "approx_tflops": round(tflops, 1), "approx_mfu_pct": round(100 * tflops / PEAK_TFLOPS[prec], 1),
            "n_params_m": round(n_params / 1e6, 1), "attn_impl": model.config._attn_implementation}

if default_dtype == torch.float32:
    print("Block 5 already used fp32 master weights -> no re-run needed")
else:
    done = done_keys(NAME_Q)
    for bs, seq, ckpt in Q_CONFIGS:
        key = f"qwen2.5-1.5b|bf16-amp|bs{bs}|seq{seq}|ckpt{int(ckpt)}"
        if key in done:
            print("skip (done):", key); continue
        row = base_row(key, model="qwen2.5-1.5b", task="train", precision="bf16", batch=bs, seq_len=seq,
                       grad_ckpt=ckpt, warmup=Q_WARMUP, iters=Q_ITERS, optimizer="AdamW(fused)",
                       master_weights="fp32", param_dtype="float32", data="synthetic",
                       init="random(from config)")
        try:
            row.update(qwen_train_fp32master(bs, seq, ckpt))
        except torch.cuda.OutOfMemoryError:
            row.update(status="OOM")
        cleanup()
        save_result(NAME_Q, row)
        print(f"{key:38s} {row['status']:4s} {row.get('tokens_per_s', '-'):>7} tok/s  "
              f"peak {row.get('peak_alloc_gb', '-')} GB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_Q}.csv")
print(df[["precision", "master_weights", "batch", "seq_len", "grad_ckpt", "status", "tokens_per_s",
          "step_ms", "peak_alloc_gb", "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]].to_string(index=False))


from_config default param dtype: torch.bfloat16
tagged 4 Block 5 rows with param_dtype=bfloat16
qwen2.5-1.5b|bf16-amp|bs4|seq512|ckpt1 ok      7926 tok/s  peak 23.95 GB  305.6 W
qwen2.5-1.5b|bf16-amp|bs8|seq512|ckpt1 ok      9995 tok/s  peak 26.73 GB  352.9 W
qwen2.5-1.5b|bf16-amp|bs4|seq1024|ckpt1 ok      9870 tok/s  peak 26.73 GB  361.3 W
qwen2.5-1.5b|bf16-amp|bs2|seq512|ckpt0 ok      6790 tok/s  peak 25.11 GB  285.6 W
precision                          master_weights  batch  seq_len  grad_ckpt status  tokens_per_s  step_ms  peak_alloc_gb  power_avg_w  tokens_per_joule  approx_mfu_pct
bf16-pure none (bf16 weights + bf16 AdamW states)      4      512       True     ok          9839    208.2          12.39        321.8              30.6            29.2
bf16-pure none (bf16 weights + bf16 AdamW states)      8      512       True     ok         12008    341.1          15.98        373.9              32.1            35.6
bf16-pure none (bf16 weights + bf16 AdamW states)      4     1024   

In [8]:
# Block 6 — HF transformers generate() baseline (Qwen2.5-1.5B-Instruct, bf16, greedy, fixed lengths)
from transformers import AutoTokenizer
import statistics

INF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
IN_LEN, OUT_LEN = 128, 256
HF_BATCHES = [1, 8, 32]
REPEATS = 2

set_tf32(True)
tok = AutoTokenizer.from_pretrained(INF_MODEL)
lm = AutoModelForCausalLM.from_pretrained(INF_MODEL, dtype=torch.bfloat16, attn_implementation="sdpa",
                                          device_map="cuda").eval()
text = "The history of high-performance computing is a story of parallelism and memory bandwidth. " * 64
base_ids = tok(text, return_tensors="pt").input_ids[0, :IN_LEN]
assert base_ids.numel() == IN_LEN

@torch.inference_mode()
def gen(ids, n_new):
    return lm.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=n_new,
                       min_new_tokens=n_new, do_sample=False, pad_token_id=tok.eos_token_id)

def timed(fn):
    t0 = time.perf_counter(); out = fn(); torch.cuda.synchronize()
    return time.perf_counter() - t0, out

def hf_bench(bs):
    ids = base_ids.repeat(bs, 1).cuda()
    gen(ids, 8); torch.cuda.synchronize()                                   # warmup
    ttft = statistics.median(timed(lambda: gen(ids, 1))[0] for _ in range(3))
    torch.cuda.reset_peak_memory_stats()
    times = []
    with GPUSampler() as s:
        for _ in range(REPEATS):
            t, out = timed(lambda: gen(ids, OUT_LEN)); times.append(t)
    assert out.shape[1] == IN_LEN + OUT_LEN, out.shape
    t, gpu = statistics.median(times), s.stats()
    out_tok_s = bs * OUT_LEN / t
    decode_tok_s = bs * (OUT_LEN - 1) / (t - ttft)
    return {"status": "ok", "ttft_ms": round(1000 * ttft, 1), "e2e_s": round(t, 2),
            "output_tokens_per_s": round(out_tok_s, 1), "decode_tokens_per_s": round(decode_tok_s, 1),
            "per_seq_decode_tok_s": round(decode_tok_s / bs, 1),
            "peak_alloc_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), **gpu,
            "tokens_per_joule": round(out_tok_s / gpu["power_avg_w"], 2) if gpu["power_avg_w"] else None}

NAME_HF = "infer_hf"
done = done_keys(NAME_HF)
for bs in HF_BATCHES:
    key = f"hf-generate|qwen2.5-1.5b-instruct|bf16|bs{bs}|in{IN_LEN}|out{OUT_LEN}"
    if key in done:
        print("skip (done):", key); continue
    row = base_row(key, engine="hf-transformers", model="qwen2.5-1.5b-instruct", task="inference",
                   precision="bf16", batch=bs, input_len=IN_LEN, output_len=OUT_LEN, decoding="greedy",
                   attn_impl="sdpa", transformers=transformers.__version__)
    try:
        row.update(hf_bench(bs))
    except torch.cuda.OutOfMemoryError:
        row.update(status="OOM")
    cleanup(); save_result(NAME_HF, row)
    print(f"bs{bs:<3} {row['status']:4s} TTFT {row.get('ttft_ms', '-')} ms  "
          f"{row.get('output_tokens_per_s', '-')} out tok/s  ({row.get('per_seq_decode_tok_s', '-')} tok/s/seq)  "
          f"peak {row.get('peak_alloc_gb', '-')} GiB  {row.get('power_avg_w', '-')} W")

df = pd.read_csv(f"{RESULTS_DIR}/{NAME_HF}.csv")
print(df[["batch", "status", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
          "peak_alloc_gb", "gpu_util_avg", "power_avg_w", "tokens_per_joule"]].to_string(index=False))

del lm; cleanup()          # free the GPU before vLLM


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

bs1   ok   TTFT 34.3 ms  33.4 out tok/s  (33.4 tok/s/seq)  peak 2.91 GiB  64.8 W
bs8   ok   TTFT 33.6 ms  258.3 out tok/s  (32.3 tok/s/seq)  peak 3.0 GiB  68.3 W
bs32  ok   TTFT 78.5 ms  1012.3 out tok/s  (31.8 tok/s/seq)  peak 3.33 GiB  79.6 W
 batch status  ttft_ms  e2e_s  output_tokens_per_s  per_seq_decode_tok_s  peak_alloc_gb  gpu_util_avg  power_avg_w  tokens_per_joule
     1     ok     34.3   7.67                 33.4                  33.4           2.91          31.2         64.8              0.52
     8     ok     33.6   7.93                258.3                  32.3           3.00          32.5         68.3              3.78
    32     ok     78.5   8.09               1012.3                  31.8           3.33          36.0         79.6             12.72


In [12]:
# Block 7a — isolated vLLM venv (main torch 2.11 env stays untouched)
!pip install -q uv
!uv venv /content/vllm-env --python 3.12 -q
!uv pip install -q --python /content/vllm-env/bin/python vllm nvidia-ml-py --torch-backend=auto
!/content/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu ok:', torch.cuda.is_available())"


? A virtual environment already exists at `vllm-env`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt^C
vllm 0.31.0 | torch 2.13.0+cu132 | cuda 13.2 | gpu ok: True


In [13]:
%%writefile /content/vllm_bench.py
# vLLM offline throughput/latency benchmark — run with the venv python, args: RESULTS_DIR
import json, os, sys, time, random, statistics, threading, datetime
import pynvml, torch, vllm
from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt

RESULTS_DIR = sys.argv[1]
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
IN_LEN, OUT_LEN = 128, 256
NUM_PROMPTS = [1, 8, 32, 128, 512]
GPU_UTIL = 0.85
OUT = f"{RESULTS_DIR}/infer_vllm.jsonl"

pynvml.nvmlInit()
_H = pynvml.nvmlDeviceGetHandleByIndex(0)

class GPUSampler:
    def __init__(self, interval=0.1):
        self.interval, self.util, self.power, self.mem = interval, [], [], []
        self._stop = threading.Event()
    def _run(self):
        while not self._stop.is_set():
            self.util.append(pynvml.nvmlDeviceGetUtilizationRates(_H).gpu)
            self.power.append(pynvml.nvmlDeviceGetPowerUsage(_H) / 1000)
            self.mem.append(pynvml.nvmlDeviceGetMemoryInfo(_H).used / 2**30)
            time.sleep(self.interval)
    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join()
    def stats(self):
        avg = lambda x: round(sum(x) / len(x), 1) if x else None
        return {"gpu_util_avg": avg(self.util), "gpu_util_max": max(self.util, default=None),
                "power_avg_w": avg(self.power), "power_max_w": round(max(self.power, default=0), 1),
                "nvml_mem_max_gb": round(max(self.mem, default=0), 2)}

def key(n): return f"vllm|qwen2.5-1.5b-instruct|bf16|n{n}|in{IN_LEN}|out{OUT_LEN}"

def main():
    done = set()
    if os.path.exists(OUT):
        done = {json.loads(l)["key"] for l in open(OUT) if l.strip()}
    todo = [n for n in NUM_PROMPTS if key(n) not in done]
    if not todo:
        print("all vLLM configs already done"); return

    t0 = time.perf_counter()
    llm = LLM(model=MODEL, dtype="bfloat16", gpu_memory_utilization=GPU_UTIL, max_model_len=1024,
              enable_prefix_caching=False, seed=0)
    load_s = time.perf_counter() - t0

    rng = random.Random(0)
    def prompts(n):   # distinct random-token prompts of exactly IN_LEN tokens
        return [TokensPrompt(prompt_token_ids=[rng.randrange(1000, 150000) for _ in range(IN_LEN)])
                for _ in range(n)]
    sp_full = SamplingParams(temperature=0.0, max_tokens=OUT_LEN, min_tokens=OUT_LEN, ignore_eos=True)
    sp_one = SamplingParams(temperature=0.0, max_tokens=1)

    llm.generate(prompts(8), sp_full, use_tqdm=False)          # warmup

    for n in todo:
        ps = prompts(n)
        ttfts = []
        for _ in range(3):
            t = time.perf_counter(); llm.generate(ps, sp_one, use_tqdm=False)
            ttfts.append(time.perf_counter() - t)
        ttft = statistics.median(ttfts)
        times = []
        with GPUSampler() as s:
            for _ in range(2):
                t = time.perf_counter(); outs = llm.generate(ps, sp_full, use_tqdm=False)
                times.append(time.perf_counter() - t)
        assert all(len(o.outputs[0].token_ids) == OUT_LEN for o in outs)
        t, gpu = statistics.median(times), s.stats()
        out_tok_s = n * OUT_LEN / t
        row = {"key": key(n), "gpu": torch.cuda.get_device_name(0), "engine": "vllm",
               "vllm": vllm.__version__, "torch": torch.__version__,
               "ts": datetime.datetime.now().isoformat(timespec="seconds"),
               "model": "qwen2.5-1.5b-instruct", "task": "inference", "precision": "bf16",
               "batch": n, "input_len": IN_LEN, "output_len": OUT_LEN, "decoding": "greedy(ignore_eos)",
               "gpu_memory_utilization": GPU_UTIL, "prefix_caching": False, "load_s": round(load_s, 1),
               "status": "ok", "ttft_ms": round(1000 * ttft, 1), "e2e_s": round(t, 2),
               "output_tokens_per_s": round(out_tok_s, 1),
               "total_tokens_per_s": round(n * (IN_LEN + OUT_LEN) / t, 1),
               "per_seq_decode_tok_s": round((OUT_LEN - 1) / (t - ttft), 1), **gpu,
               "tokens_per_joule": round(out_tok_s / gpu["power_avg_w"], 2) if gpu["power_avg_w"] else None}
        with open(OUT, "a") as f:
            f.write(json.dumps(row) + "\n"); f.flush(); os.fsync(f.fileno())
        print(f"n={n:<4} TTFT {row['ttft_ms']} ms  {row['output_tokens_per_s']} out tok/s  "
              f"({row['per_seq_decode_tok_s']} tok/s/seq)  {gpu['power_avg_w']} W")

if __name__ == "__main__":
    main()


Overwriting /content/vllm_bench.py


In [14]:
# Block 7c — run vLLM benchmark in the venv, then build CSV + table in the main env
VENV_PY = "/content/vllm-env/bin/python"
!{VENV_PY} /content/vllm_bench.py "{RESULTS_DIR}"

rows = [json.loads(l) for l in open(f"{RESULTS_DIR}/infer_vllm.jsonl") if l.strip()]
df = pd.DataFrame(rows)
df.to_csv(f"{RESULTS_DIR}/infer_vllm.csv", index=False)
print(df[["batch", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
          "gpu_util_avg", "power_avg_w", "tokens_per_joule", "load_s"]].to_string(index=False))


(MainProcess pid=21492) INFO 10-14 00:51:58 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 1024, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=21492) INFO 10-14 00:51:58 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=21492) INFO 10-14 00:51:58 [model.py:2109] Using max model len 1024
(MainProcess pid=21492) INFO 10-14 00:51:58 [scheduler.py:308] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=21492) INFO 10-14 00:51:59 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=21588) INFO 10-14 00:52:02 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct',

FileNotFoundError: [Errno 2] No such file or directory: '/content/gpu_bench/a100_40gb/infer_vllm.jsonl'

In [15]:
# Block 7d — back up results, free GPU memory, re-run vLLM with the full log saved
# 1) Results live in /content only (Drive not mounted) -> download a backup now
!cd /content/gpu_bench && zip -qr /content/a100_partial_backup.zip a100_40gb
from google.colab import files; files.download("/content/a100_partial_backup.zip")

# 2) What is holding GPU memory?
gc.collect(); torch.cuda.empty_cache()
print(f"main kernel torch: allocated {torch.cuda.memory_allocated()/2**30:.2f} GiB, "
      f"reserved {torch.cuda.memory_reserved()/2**30:.2f} GiB")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!pkill -f /content/vllm-env/bin/python; pkill -f EngineCore; sleep 3
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

# 3) Re-run with a lower memory reservation (0.6); log to file, print only the error lines
!sed -i 's/^GPU_UTIL = 0.85$/GPU_UTIL = float(sys.argv[2]) if len(sys.argv) > 2 else 0.85/' /content/vllm_bench.py
!grep -n "^GPU_UTIL" /content/vllm_bench.py
!{VENV_PY} /content/vllm_bench.py "{RESULTS_DIR}" 0.6 > /content/vllm_log.txt 2>&1; echo "exit code: $?"
!grep -nE "Error|error|Exception|Traceback|Free memory" /content/vllm_log.txt | grep -v " INFO " | head -40
!tail -n 8 /content/vllm_log.txt

p = f"{RESULTS_DIR}/infer_vllm.jsonl"
if os.path.exists(p):
    df = pd.DataFrame([json.loads(l) for l in open(p) if l.strip()])
    df.to_csv(f"{RESULTS_DIR}/infer_vllm.csv", index=False)
    print(df[["batch", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
              "gpu_util_avg", "power_avg_w", "tokens_per_joule", "load_s"]].to_string(index=False))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

main kernel torch: allocated 0.02 GiB, reserved 0.04 GiB
memory.used [MiB], memory.total [MiB]
628 MiB, 40960 MiB
^C
memory.used [MiB], memory.total [MiB]
628 MiB, 40960 MiB
11:GPU_UTIL = float(sys.argv[2]) if len(sys.argv) > 2 else 0.85
exit code: 1
39:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433] Traceback (most recent call last):
135:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433] FileNotFoundError: [Errno 2] No such file or directory: 'ninja'
137:Traceback (most recent call last):
239:FileNotFoundError: [Errno 2] No such file or directory: 'ninja'
241:Traceback (most recent call last):
271:    raise RuntimeError(
272:RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}
    with launch_core_engines(
  File "/usr/lib/python3.12/contextlib.py", line 144, in __exit__
    next(self.gen)
  File "/content/vllm-env/lib/python3.12/site-packages/vllm/v1/engine/utils.py", line 1268, in launch_core_engines
    wait_for_engine_st

In [16]:
# Block 7e — fix missing `ninja` (FlashInfer JIT), retry vLLM
!cp /content/vllm_log.txt /content/vllm_log_fail_ninja.txt
!grep -nE "flashinfer|jit|ninja" /content/vllm_log_fail_ninja.txt | grep -v " INFO " | head -8

!uv pip install -q --python /content/vllm-env/bin/python ninja
!/content/vllm-env/bin/ninja --version; which nvcc && nvcc --version | tail -n 1

!PATH=/content/vllm-env/bin:$PATH VLLM_USE_FLASHINFER_SAMPLER=0 {VENV_PY} /content/vllm_bench.py "{RESULTS_DIR}" 0.6 > /content/vllm_log.txt 2>&1; echo "exit code: $?"
!grep -nE "Error|Exception" /content/vllm_log.txt | grep -v " INFO " | head -20
!tail -n 8 /content/vllm_log.txt

p = f"{RESULTS_DIR}/infer_vllm.jsonl"
if os.path.exists(p):
    df = pd.DataFrame([json.loads(l) for l in open(p) if l.strip()])
    df.to_csv(f"{RESULTS_DIR}/infer_vllm.csv", index=False)
    print(df[["batch", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
              "gpu_util_avg", "power_avg_w", "tokens_per_joule", "load_s"]].to_string(index=False))


102:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]     sampled = flashinfer_sample(processed_logits, top_k, top_p).to(
104:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]   File "/content/vllm-env/lib/python3.12/site-packages/vllm/v1/sample/ops/topk_topp_sampler.py", line 602, in flashinfer_sample
105:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]     next_token_ids = flashinfer.sampling.top_k_top_p_sampling_from_logits(
107:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]   File "/content/vllm-env/lib/python3.12/site-packages/flashinfer/api_logging.py", line 2348, in _auto_dump_wrapper
110:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]   File "/content/vllm-env/lib/python3.12/site-packages/flashinfer/sampling.py", line 1547, in top_k_top_p_sampling_from_logits
113:(EngineCore pid=22309) ERROR 10-14 00:54:38 [core.py:1433]   File "/content/vllm-env/lib/python3.12/site-packages/flashinfer/api_logging.py", line 2348, in _auto_dump_

In [17]:
# Block 7f — retry vLLM via subprocess (no IPython $/{} expansion pitfalls)
VENV_PY = "/content/vllm-env/bin/python"
env = {**os.environ,
       "PATH": "/content/vllm-env/bin:" + os.environ["PATH"],   # ninja visible to JIT builds
       "VLLM_USE_FLASHINFER_SAMPLER": "0"}                       # greedy decoding -> sampler not used

with open("/content/vllm_log.txt", "w") as log:
    rc = subprocess.run([VENV_PY, "/content/vllm_bench.py", RESULTS_DIR, "0.6"],
                        stdout=log, stderr=subprocess.STDOUT, env=env).returncode
print("exit code:", rc)

!grep -nE "Error|Exception" /content/vllm_log.txt | grep -v " INFO " | head -20
!grep -E "^n=" /content/vllm_log.txt
!tail -n 5 /content/vllm_log.txt

p = f"{RESULTS_DIR}/infer_vllm.jsonl"
if os.path.exists(p):
    df = pd.DataFrame([json.loads(l) for l in open(p) if l.strip()])
    df.to_csv(f"{RESULTS_DIR}/infer_vllm.csv", index=False)
    print(df[["batch", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
              "gpu_util_avg", "power_avg_w", "tokens_per_joule", "load_s"]].to_string(index=False))


exit code: 0
n=1    TTFT 13.1 ms  238.7 out tok/s  (240.8 tok/s/seq)  200.9 W
n=8    TTFT 37.9 ms  1908.6 out tok/s  (246.3 tok/s/seq)  207.6 W
n=32   TTFT 78.9 ms  6708.6 out tok/s  (223.3 tok/s/seq)  233.6 W
n=128  TTFT 248.5 ms  19151.3 out tok/s  (174.4 tok/s/seq)  308.3 W
n=512  TTFT 883.7 ms  25384.5 out tok/s  (59.6 tok/s/seq)  351.4 W
(MainProcess pid=23307) INFO 10-14 00:59:41 [utils.py:638] [shutdown] Process manager: send sigterm to process EngineCore
(EngineCore pid=23410) INFO 10-14 00:59:41 [core.py:1409] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=23410) INFO 10-14 00:59:41 [core.py:1558] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=23410) INFO 10-14 00:59:41 [core.py:1589] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=23410) INFO 10-14 00:59:41 [core.py:1422] [shutdown] EngineCore: exiting busy loop
 batch  ttft_ms  e2e_s  output_tokens_per_s  per_seq_decode_tok_s  gpu_util

In [18]:
# Block 8 — all_results.csv, summary.md, copy to Drive, zip + download
import shutil
from google.colab import files
try:
    import tabulate                       # needed by DataFrame.to_markdown
except ImportError:
    !pip install -q tabulate

R = RESULTS_DIR
env = json.load(open(f"{R}/env.json"))
gpt2 = pd.read_csv(f"{R}/train_gpt2.csv")
rn   = pd.read_csv(f"{R}/train_resnet50.csv")
qw   = pd.read_csv(f"{R}/train_qwen2.5-1.5b.csv")
hf   = pd.read_csv(f"{R}/infer_hf.csv")
vl   = pd.read_csv(f"{R}/infer_vllm.csv")

# 1) combined long-format CSV
pd.concat([d.assign(benchmark=n) for n, d in [("train_gpt2", gpt2), ("train_resnet50", rn),
           ("train_qwen2.5-1.5b", qw), ("infer_hf", hf), ("infer_vllm", vl)]],
          ignore_index=True).to_csv(f"{R}/all_results.csv", index=False)

# 2) reproducibility files
for src in ["/content/vllm_bench.py", "/content/vllm_log.txt", "/content/vllm_log_fail_ninja.txt"]:
    if os.path.exists(src): shutil.copy(src, R)

# 3) summary.md
def md(df, cols): return df[[c for c in cols if c in df.columns]].to_markdown(index=False)
def val(df, q, col): return df.query(q)[col].iloc[0]

TRAIN_COLS = ["precision", "batch", "status", "tokens_per_s", "step_ms", "peak_alloc_gb",
              "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]
RN_COLS = ["precision", "batch", "status", "images_per_s", "step_ms", "peak_alloc_gb",
           "power_avg_w", "images_per_joule", "approx_mfu_pct"]
Q_COLS = ["precision", "master_weights", "batch", "seq_len", "grad_ckpt", "status", "tokens_per_s",
          "step_ms", "peak_alloc_gb", "power_avg_w", "tokens_per_joule", "approx_mfu_pct"]
INF_COLS = ["engine", "batch", "ttft_ms", "e2e_s", "output_tokens_per_s", "per_seq_decode_tok_s",
            "gpu_util_avg", "power_avg_w", "tokens_per_joule"]

cmp_rows = [
    ("GPT-2 124M train, strict FP32, bs16, seq1024 (tok/s)", val(gpt2, "precision=='fp32' and batch==16", "tokens_per_s")),
    ("GPT-2 124M train, BF16 AMP, bs16, seq1024 (tok/s)", val(gpt2, "precision=='bf16' and batch==16", "tokens_per_s")),
    ("GPT-2 124M train, FP16 AMP, bs8, seq1024 (tok/s)", val(gpt2, "precision=='fp16' and batch==8", "tokens_per_s")),
    ("ResNet-50 train, strict FP32, bs64 (img/s)", val(rn, "precision=='fp32' and batch==64", "images_per_s")),
    ("ResNet-50 train, FP16 AMP, bs64 (img/s)", val(rn, "precision=='fp16' and batch==64", "images_per_s")),
    ("ResNet-50 train, FP16 AMP, bs256 (img/s)", val(rn, "precision=='fp16' and batch==256", "images_per_s")),
    ("Qwen2.5-1.5B train, BF16 AMP + FP32 master, bs8, seq512, ckpt (tok/s)",
     val(qw, "precision=='bf16' and batch==8 and grad_ckpt", "tokens_per_s")),
    ("HF generate, Qwen2.5-1.5B-Instruct, bs1 (out tok/s)", val(hf, "batch==1", "output_tokens_per_s")),
    ("vLLM, Qwen2.5-1.5B-Instruct, bs1 (out tok/s)", val(vl, "batch==1", "output_tokens_per_s")),
    ("vLLM, Qwen2.5-1.5B-Instruct, 512 concurrent (out tok/s)", val(vl, "batch==512", "output_tokens_per_s")),
]
cmp = pd.DataFrame([{"Benchmark": b, "A100-SXM4-40GB": v, "RTX A6000": "TBD", "TITAN RTX": "TBD",
                     "GTX 1080 Ti": "TBD", "TITAN Xp": "TBD"} for b, v in cmp_rows])

vllm_meta = json.loads(open(f"{R}/infer_vllm.jsonl").readline())
L = [
 f"# Single-GPU reference benchmarks — {env['gpu']} (Google Colab)", "",
 "## Environment", "",
 f"- GPU: {env['gpu']}, {env['vram_gb']} GiB, compute capability {env['capability']}, {env['sm_count']} SMs",
 f"- nvidia-smi: `{env['nvidia_smi']}`",
 f"- Training / HF inference: torch {env['torch']} (CUDA {env['cuda']}, cuDNN {env['cudnn']}), "
 f"transformers {env['transformers']}, Python {env['python']}",
 f"- vLLM inference (separate venv): vLLM {vllm_meta['vllm']}, torch {vllm_meta['torch']}, Python 3.12",
 f"- Run date: {env['timestamp'][:10]}", "",
 "## Method", "",
 "- Synthetic inputs generated on the GPU (no dataloader / disk I/O). Fixed seeds.",
 "- Training: 10 warmup + 40 timed steps (Qwen: 5 + 20). Full optimizer step included.",
 "- `fp32` = strict IEEE FP32 (TF32 off); `tf32` = FP32 with TF32 matmul/conv; `fp16`/`bf16` = autocast AMP "
 "(FP16 uses GradScaler).",
 "- GPU util / power / memory sampled via NVML every 100 ms during the timed region. "
 "`peak_alloc_gb` = `torch.cuda.max_memory_allocated` (GiB).",
 "- `approx_mfu_pct` = 6·N·tokens/s (ResNet: 3×fwd FLOPs) vs A100 dense peak "
 "(FP32 19.5, TF32 156, FP16/BF16 312 TFLOPS); excludes attention FLOPs and recompute.",
 "- Inference: 128 input tokens, exactly 256 output tokens, greedy decoding. "
 "TTFT = median time of a 1-token generate for the whole batch.", "",
 "## Training", "",
 "### GPT-2 124M (seq 1024, AdamW fused, SDPA attention)", "", md(gpt2, TRAIN_COLS), "",
 "### ResNet-50 (224×224, channels_last, SGD momentum)", "", md(rn, RN_COLS), "",
 "### Qwen2.5-1.5B (random init from config, AdamW fused, SDPA)", "",
 "`bf16-pure` = BF16 weights and BF16 optimizer states (memory-lean, not the standard recipe). "
 "`bf16` + `fp32` master = standard mixed-precision training.", "", md(qw, Q_COLS), "",
 "## Inference — Qwen2.5-1.5B-Instruct, BF16", "",
 md(pd.concat([hf, vl], ignore_index=True), INF_COLS), "",
 "## Key findings", "",
 "1. **Transformers train fastest in BF16.** GPT-2: 66.3k tok/s (BF16) vs 19.0k (strict FP32); "
 "4.2× more tokens per joule. TF32 alone gives a free 2.7× over strict FP32.",
 "2. **CNNs prefer FP16 AMP over BF16 on A100** (ResNet-50: 2,278 vs 1,944 img/s at bs256, +17%), "
 "while for GPT-2 the two are tied. FP16 AMP is 4.8× more energy-efficient than FP32.",
 "3. **40 GB is comfortable for full fine-tuning of ~1.5B models.** Qwen2.5-1.5B with FP32 master weights: "
 "10.0k tok/s at 26.7 GiB peak (~30% MFU). Pure-BF16 is 20% faster and fits in 16 GiB.",
 "4. **Small-model eager training is overhead-bound.** GPT-2 BF16 reaches only ~16% MFU "
 "(LayerNorm/GELU/dropout/50k-vocab loss/launch overhead); the larger Qwen reaches ~30%.",
 "5. **Serving engine matters more than the GPU for decode.** HF `generate` is CPU/launch-bound "
 "(~33 tok/s per sequence regardless of batch, GPU util ~33%, 65–80 W). vLLM is 7.1× faster at bs1 "
 "(239 tok/s, ~48% of the memory-bandwidth roofline) and peaks at 25.4k out tok/s with 512 concurrent "
 "requests (72 tok/J).",
 "6. **Batch saturation points:** GPT-2 saturates at bs8–16 (+3% from 8→16), ResNet-50 AMP at ~bs128–256, "
 "vLLM at ~128–512 concurrent requests.", "",
 "## Caveats", "",
 "- Single run per config; expect a few % run-to-run variance. Colab A100 is a shared host.",
 "- Synthetic data: numbers exclude input pipeline; real dataloaders (esp. ImageNet JPEG decode) will be lower.",
 "- NVML `gpu_util` only measures the fraction of time any kernel runs — not SM occupancy. Power is more informative.",
 "- HF inference numbers depend strongly on host CPU and library versions; compare across machines with care.",
 f"- vLLM: `gpu_memory_utilization={vllm_meta['gpu_memory_utilization']}`, prefix caching off, "
 "FlashInfer sampler disabled (`VLLM_USE_FLASHINFER_SAMPLER=0`; irrelevant for greedy decoding). "
 f"Engine startup ({vllm_meta['load_s']} s) excluded. Per-seq decode at n=512 is not meaningful (queueing).",
 "- vLLM ran with a newer torch (separate venv) than the training/HF benchmarks.",
 "- GPT-2 bs32 @ seq1024 OOMs at all precisions (50k-vocab logits + activations); throughput had already saturated at bs16.", "",
 "## Cross-GPU comparison (fill in from previous runs)", "",
 "Only compare rows whose config matches exactly (model, precision, batch, seq/img size, synthetic vs real data). "
 "Pascal GPUs (1080 Ti, TITAN Xp, CC 6.1) have no tensor cores or BF16 and are not supported by vLLM; "
 "TITAN RTX (CC 7.5) supports FP16 tensor cores but not BF16.", "",
 cmp.to_markdown(index=False), "",
]
open(f"{R}/summary.md", "w").write("\n".join(L))
print(f"wrote {R}/summary.md and all_results.csv")

# 4) copy to Drive (if not already there), 5) zip + download
if not R.startswith("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dst = "/content/drive/MyDrive/gpu_bench/a100_40gb"
        shutil.copytree(R, dst, dirs_exist_ok=True); print("copied to Drive:", dst)
    except Exception as e:
        print("Drive copy failed — use the zip download:", e)
shutil.make_archive("/content/a100_40gb_results", "zip", R)
files.download("/content/a100_40gb_results.zip")
print(sorted(os.listdir(R)))


wrote /content/gpu_bench/a100_40gb/summary.md and all_results.csv
Drive copy failed — use the zip download: [dfs_ephemeral] Credentials propagation unsuccessful: the server still reports the credentials as unauthorized


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['all_results.csv', 'env.json', 'infer_hf.csv', 'infer_hf.jsonl', 'infer_vllm.csv', 'infer_vllm.jsonl', 'summary.md', 'train_gpt2.csv', 'train_gpt2.jsonl', 'train_qwen2.5-1.5b.csv', 'train_qwen2.5-1.5b.jsonl', 'train_resnet50.csv', 'train_resnet50.jsonl', 'vllm_bench.py', 'vllm_log.txt', 'vllm_log_fail_ninja.txt']


In [19]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
svc = build("drive", "v3")
f = svc.files().create(body={"name": "a100_40gb_results.zip"},
                       media_body=MediaFileUpload("/content/a100_40gb_results.zip", mimetype="application/zip"),
                       fields="id,webViewLink").execute()
print("Uploaded to My Drive (root):", f["webViewLink"])


Uploaded to My Drive (root): https://drive.google.com/file/d/1VT_JW6tosDsNFzcrxjQbFODYHXXFDOfb/view?usp=drivesdk
